# Lab 70: The Agent Detector

*Brain Lab: 84 Experiments to Decode Your Biological and Artificial Intelligence* · Chapter 8, *The Social Network*

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/labphonlab/brain-lab-support/blob/main/notebooks/lab-70.ipynb)

**What you will do:** code your own and your friends' descriptions of the Heider-Simmel animation for intentional language and agreement on the roles, then feed the same geometric scene to a small AI language model and a rebuilt ELIZA to see how differently they "see" agency.

**How to use.** Run the cells from top to bottom (Shift+Enter). Nothing needs installing, and nothing you do here can break anything: *Runtime → Restart and run all* returns it to the start. To keep your own results, use *File → Save a copy in Drive*.

In [ ]:
# Setup: run this cell first (Shift+Enter).
import platform, numpy as np, pandas as pd, matplotlib
import matplotlib.pyplot as plt
np.random.seed(2026)
print("Python", platform.python_version(), "| numpy", np.__version__, "| pandas", pd.__version__)

## The experiment in the book

In Lab 70 you watched the Heider and Simmel (1944) animation of two triangles and a circle
moving around a rectangle, with no faces, sound, or dialogue. You wrote down your own
description, then compared it with friends' descriptions. The book reports remarkable
agreement on the broad narrative — a bully, a victim, and a rescuer — even though the
animation contains no explicit social content at all.

## Record your results

Code each description (yours and your friends') for the role each shape was given, and
whether the language used was intentional or emotional (chasing, protecting, angry, scared)
rather than purely geometric (moved, turned, approached).

In [ ]:
# example data: replace with yours
narratives = pd.DataFrame({
    "person": ["You", "Friend 1", "Friend 2", "Friend 3"],
    "role_big_triangle": ["bully", "bully", "aggressor", "bully"],
    "role_small_triangle": ["victim", "victim", "victim", "scared"],
    "role_circle": ["rescuer", "protector", "rescuer", "rescuer"],
    "used_intentional_language": [True, True, True, True],
})
display(narratives)
pct_intentional = narratives.used_intentional_language.mean() * 100
print(f"{pct_intentional:.0f}% of descriptions used intentional or emotional language rather")
print("than purely geometric terms.")

## Compare

Lab 70 claims "remarkable agreement" on the broad narrative, without giving an exact
percentage. Check how much your group actually agreed on who played which role.

In [ ]:
agree_big = narratives.role_big_triangle.str.contains("bully|aggressor").mean() * 100
agree_small = narratives.role_small_triangle.str.contains("victim|scared").mean() * 100
agree_circle = narratives.role_circle.str.contains("rescuer|protector").mean() * 100
print(f"Agreement that the big triangle is the aggressor: {agree_big:.0f}%")
print(f"Agreement that the small triangle is the victim: {agree_small:.0f}%")
print(f"Agreement that the circle is the rescuer: {agree_circle:.0f}%")
print("Lab 70 does not give a precise number for 'remarkable agreement' -- treat high")
print("agreement here as support for the claim, not a replication of a specific figure.")

## The AI side

The book connects this to the **ELIZA effect**: people attribute understanding to systems
that only pattern-match (Weizenbaum, 1976). Below, a small open language model is given a
purely geometric description of the animation and asked what is happening, exactly as the
book's "AI extension" suggests, alongside a miniature rebuilt ELIZA for contrast.

*This part downloads a small open model or library, so it runs in Colab (it may take a minute the first time).*

In [ ]:
from transformers import pipeline
generator = pipeline("text-generation", model="distilgpt2")

geometric_description = (
    "A large triangle moves toward a small triangle. The small triangle moves away. "
    "A circle moves between them. Question: what is happening in this scene? Answer:"
)
out = generator(geometric_description, max_new_tokens=40, do_sample=True,
                 temperature=0.8, num_return_sequences=1)[0]["generated_text"]
print(out)

intentional_words = ["chas", "flee", "escap", "protect", "attack", "afraid", "angry", "help", "bull", "rescu"]
found = [w for w in intentional_words if w in out.lower()]
print("\nIntentional or emotional words found:", found or "none")

# A miniature ELIZA-style rule-based responder, for contrast with the LLM above
import re
ELIZA_RULES = [
    (r"\bcircle\b", "Tell me more about the circle."),
    (r"\btriangle\b", "Why do you think the triangle behaves that way?"),
    (r"\b(chase|chasing|toward)\b", "That sounds threatening. How does that make you feel?"),
]
def eliza_reply(text):
    for pattern, reply in ELIZA_RULES:
        if re.search(pattern, text, re.I):
            return reply
    return "Please tell me more."
print("\nELIZA:", eliza_reply(geometric_description))

The language model's continuation often uses exactly the intentional vocabulary humans reach
for (chasing, fleeing, protecting) — not because it understands agency, but because its
training text is full of human descriptions of similar motion using that vocabulary, so
predicting the next likely word after a chase-like description tends to produce chase-like
words. ELIZA cannot even parse the geometry: it just matches keywords in your prompt back
into a canned response. Both are worth comparing with your own account: yours came from a
spontaneous, effortless perception of agency; theirs came from statistical association or
simple pattern-matching, with no percept behind it at all.

## Pool the class data

Pool the class's coded descriptions to see how consistently people across a larger group
reach for intentional language and agree on the same roles.

In [ ]:
import io
csv_text = """id,used_intentional_language,agreed_big_triangle_aggressor
P01,1,1
P02,1,1
P03,1,0
P04,1,1
P05,0,1
P06,1,1
"""  # example data: replace with your class CSV
class_df = pd.read_csv(io.StringIO(csv_text))
pct_intentional = class_df.used_intentional_language.mean() * 100
pct_agree = class_df.agreed_big_triangle_aggressor.mean() * 100
boot = [class_df.agreed_big_triangle_aggressor.sample(len(class_df), replace=True).mean() * 100
        for _ in range(5000)]
lo, hi = np.percentile(boot, [2.5, 97.5])
print(f"{pct_intentional:.0f}% used intentional or emotional language.")
print(f"{pct_agree:.0f}% agreed the big triangle was the aggressor (95% bootstrap interval {lo:.0f}-{hi:.0f}%)")

## Questions to think about

1. The book's hyperactive agency detection device (HADD) argues that over-detecting agency is evolutionarily safer than under-detecting it. Can you think of a moving pattern in nature (wind in leaves, water over rocks) that never triggers an agency percept for you, and one that reliably does?
2. Did the language model's continuation above ever contradict itself (for example, calling the small triangle both a victim and an aggressor within the same sentence)? What would that suggest about whether it is tracking a consistent scene, or generating locally plausible text one word at a time?
3. The book connects the ELIZA effect to people naming their cars and apologising to their Roombas. Have you ever caught yourself doing something similar with an object or a chatbot, and did knowing the mechanism (pattern completion, not real agency) change how you felt about it afterwards?
4. Conspiracy theories are linked in the text to the same agency-detection system misfiring on complex, impersonal events. Based on what you saw with the geometric shapes, why might an agentive explanation feel more satisfying than a "no one is in control" explanation, even when it is wrong?

## Challenge

Describe a different short video (a nature documentary clip, a sports highlight, a screensaver)
in purely geometric terms — shapes, positions, movements, no character names or motives — and
ask a friend to guess what is "really" happening from your description alone. See how much
agency they infer from geometry you tried hard to strip out.

## Record what you ran

In [ ]:
NOTEBOOK_ID = "lab-70.ipynb"
# Reproducibility record: paste this output into your notes or report.
import sys, platform, datetime, numpy, pandas, matplotlib
print("Notebook:", NOTEBOOK_ID)
print("Run at  :", datetime.datetime.now().isoformat(timespec="seconds"))
print("Python  :", sys.version.split()[0], "on", platform.platform())
print("numpy", numpy.__version__, "| pandas", pandas.__version__, "| matplotlib", matplotlib.__version__)
print("Seed    : 2026")